In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import plopp as pp
import scipp as sc
from chemformula import ChemFormula

from redcamel import RemiCalculator, sample_photoionization

In [ ]:
length_acceleration_ion = sc.scalar(0.108, unit="m")
length_drift_ion = sc.scalar(0.0, unit="m")
voltage_ion = sc.scalar(-13.628, unit="V")
length_acceleration_electron = sc.scalar(0.195, unit="m")
length_drift_electron = sc.scalar(0.0, unit="m")
voltage_electron = sc.scalar(24.8, unit="V")
magnetic_field = sc.scalar(-6.503, unit="G")
v_jet = sc.scalar(994.0, unit="m/s")
jet_direction = "+x"
field_direction = "+z"
projectile_direction = "-y"

In [ ]:
real_remi = RemiCalculator(
    length_acceleration_ion=length_acceleration_ion,
    length_drift_ion=length_drift_ion,
    voltage_ion=voltage_ion,
    length_acceleration_electron=length_acceleration_electron,
    length_drift_electron=length_drift_electron,
    voltage_electron=voltage_electron,
    magnetic_field=magnetic_field,
    v_jet=v_jet,
    jet_direction=jet_direction,
    field_direction=field_direction,
    projectile_direction=projectile_direction,
    resolution_x=sc.scalar(0.2, unit="mm"),
    resolution_y=sc.scalar(0.2, unit="mm"),
    resolution_tof=sc.scalar(0.1, unit="ns"),
)
perfect_remi = RemiCalculator(
    length_acceleration_ion=length_acceleration_ion,
    length_drift_ion=length_drift_ion,
    voltage_ion=voltage_ion,
    length_acceleration_electron=length_acceleration_electron,
    length_drift_electron=length_drift_electron,
    voltage_electron=voltage_electron,
    magnetic_field=magnetic_field,
    v_jet=v_jet,
    jet_direction=jet_direction,
    field_direction=field_direction,
    projectile_direction=projectile_direction,
    resolution_x=sc.scalar(0.0, unit="mm"),
    resolution_y=sc.scalar(0.0, unit="mm"),
    resolution_tof=sc.scalar(0.0, unit="ns"),
)

In [ ]:
real_coincidence = sample_photoionization(
    atom_formula=ChemFormula("He"),
    binding_energy=sc.scalar(25.0, unit="eV"),
    photon_energy=sc.scalar(30.0, unit="eV"),
    energy_width=sc.scalar(0.2, unit="eV"),
    sizes={"pulses": 100_000, "p": 1},
    remi=real_remi,
)
real_coincidence.calculate_detector_hits()
real_coincidence.calculate_momenta()

perfect_coincidence = sample_photoionization(
    atom_formula=ChemFormula("He"),
    binding_energy=sc.scalar(25.0, unit="eV"),
    photon_energy=sc.scalar(30.0, unit="eV"),
    energy_width=sc.scalar(0.2, unit="eV"),
    sizes={"pulses": 100_000, "p": 1},
    remi=perfect_remi,
)
perfect_coincidence.calculate_detector_hits()
perfect_coincidence.calculate_momenta()

In [ ]:
ion_hits_real = real_coincidence.detector_hits["He"]
ion_hits_perfect = perfect_coincidence.detector_hits["He"]

In [ ]:
real_coincidence.detector_hits["He"]

In [ ]:
ion_hits_real.coords["x"].max()

In [ ]:
x_tof_hist_real = ion_hits_real.hist(x=100, tof=100, dim=("p", "pulses"))
x_tof_hist_real.plot(norm="log", cmap="PuBuGn", title="finite position resolution")

In [ ]:
pos_bins = 100
x_bins = sc.linspace(
    "x", ion_hits_real.coords["R"].min() * 0.9, ion_hits_real.coords["R"].max() * 1.1, pos_bins
)
tof_bins = sc.linspace(
    "tof", ion_hits_real.coords["tof"].min() * 0.99, ion_hits_real.coords["tof"].max() * 1.01, 1000
)

x_tof_hist_real = ion_hits_real.hist(x=x_bins, tof=tof_bins, dim=("p", "pulses"))
real_plot = x_tof_hist_real.plot(norm="log", cmap="PuBuGn", title="finite position resolution")

x_tof_hist_perfect = ion_hits_perfect.hist(x=x_bins, tof=tof_bins, dim=("p", "pulses"))
perfect_plot = x_tof_hist_perfect.plot(norm="log", cmap="PuBuGn", title="zero position resolution")

real_plot / perfect_plot

In [ ]:
tof_bins = sc.linspace("tof", 1.5e4, 1.7e4, 1000, unit="ns")

tof_hist_real = ion_hits_real.hist(tof=tof_bins, dim=("p", "pulses"))
tof_hist_perfect = ion_hits_perfect.hist(tof=tof_bins, dim=("p", "pulses"))
pp.plot({"real": tof_hist_real, "perfect": tof_hist_perfect})

In [ ]:
x_bins = sc.linspace("x", 5, 25, 1000, unit="mm")

x_hist_real = ion_hits_real.hist(x=x_bins, dim=("p", "pulses"))
x_hist_perfect = ion_hits_perfect.hist(x=x_bins, dim=("p", "pulses"))
pp.plot({"real": x_hist_real, "perfect": x_hist_perfect})

In [ ]:
ion_momenta_real = real_coincidence.momenta["He"]
ion_momenta_perfect = perfect_coincidence.momenta["He"]

In [ ]:
ion_momenta_real

In [ ]:
p_z_bins = sc.linspace("p_z", -2, 2, 1000, unit="au momentum")

p_z_hist_real = ion_momenta_real.hist(p_z=p_z_bins, dim=("p", "pulses"))
p_z_hist_perfect = ion_momenta_perfect.hist(p_z=p_z_bins, dim=("p", "pulses"))
pp.plot({"real": p_z_hist_real, "perfect": p_z_hist_perfect})

In [ ]:
real_momentum_sum = real_coincidence.momentum_sum
perfect_momentum_sum = perfect_coincidence.momentum_sum

In [ ]:
p_z_bins = sc.linspace("p_z", -2, 2, 1000, unit="au momentum")

p_z_hist_real = real_momentum_sum.hist(p_z=p_z_bins, dim=("p", "pulses"))
p_z_hist_perfect = perfect_momentum_sum.hist(p_z=p_z_bins, dim=("p", "pulses"))
pp.plot({"real": p_z_hist_real, "perfect": p_z_hist_perfect})

In [ ]:
p_x_bins = sc.linspace("p_x", -2, 2, 1000, unit="au momentum")

p_x_hist_real = real_momentum_sum.hist(p_x=p_x_bins, dim=("p", "pulses"))
p_x_hist_perfect = perfect_momentum_sum.hist(p_x=p_x_bins, dim=("p", "pulses"))
pp.plot({"real": p_x_hist_real, "perfect": p_x_hist_perfect})

In [ ]:
electron_momenta_real = real_coincidence.momenta["e"]
electron_momenta_perfect = perfect_coincidence.momenta["e"]

In [ ]:
energy_bins = sc.linspace("energy", 0, 10.0, 1000, unit="eV")

energy_hist_real = electron_momenta_real.hist(energy=energy_bins, dim=("p", "pulses"))
energy_hist_perfect = electron_momenta_perfect.hist(energy=energy_bins, dim=("p", "pulses"))
pp.plot({"real": energy_hist_real, "perfect": energy_hist_perfect})

In [ ]:
test_remi = RemiCalculator(
    length_acceleration_ion=length_acceleration_ion,
    length_drift_ion=length_drift_ion,
    voltage_ion=voltage_ion,
    length_acceleration_electron=length_acceleration_electron,
    length_drift_electron=length_drift_electron,
    voltage_electron=voltage_electron,
    magnetic_field=magnetic_field,
    v_jet=v_jet,
    jet_direction=jet_direction,
    field_direction=field_direction,
    projectile_direction=projectile_direction,
    resolution_x=sc.scalar(0.5, unit="mm"),
    resolution_y=sc.scalar(0.4, unit="mm"),
    resolution_tof=sc.scalar(0.1, unit="ns"),
)

In [ ]:
binding_energy = sc.scalar(72.0, unit="eV")
photon_energies = sc.arange("PE", 72.5, 80.0, 1.5, unit="eV")
bandwidth = sc.scalar(0.1, unit="eV")
coincidence_list = [
    sample_photoionization(
        atom_formula=ChemFormula("He"),
        binding_energy=binding_energy,
        photon_energy=photon_energy,
        energy_width=bandwidth,
        sizes={"pulses": 100_000, "p": 1},
        remi=test_remi,
    )
    for photon_energy in photon_energies
]
for coin in coincidence_list:
    coin.calculate_detector_hits()
    coin.calculate_momenta()
    coin.set_wiggle_mask(threshold=0.5)
electron_collection = sc.concat([coin.momenta["e"] for coin in coincidence_list], dim="PE")
electron_collection.coords["PE"] = photon_energies
electron_collection.coords["energy"].nanstd(dim=["pulses", "p"], ddof=1)
electron_collection

In [ ]:
electron_collection

In [ ]:
R_limit = sc.scalar(30.0, unit="mm")
pos_bins = 100
x_bins = sc.linspace("x", -R_limit, R_limit, pos_bins)
R_bins = sc.linspace("R", 0 * R_limit, R_limit, pos_bins)
tof_bins = sc.linspace("tof", 0e2, 8e2, 1000, unit="ns")

x_tof_hist = electron_collection.hist(x=x_bins, tof=tof_bins, dim=("p", "pulses", "PE"))
x_tof_plot = x_tof_hist.plot(norm="log", cmap="PuBuGn")

R_tof_hist = electron_collection.hist(R=R_bins, tof=tof_bins, dim=("p", "pulses", "PE"))
R_tof_plot = R_tof_hist.plot(norm="log", cmap="PuBuGn")

x_tof_plot / R_tof_plot

In [ ]:
plotthing = pp.plot(
    {
        f"{energy:c} +- {bandwidth:c}": coin.momenta["e"].hist(
            energy=energy_bins, dim=("p", "pulses")
        )
        for energy, coin in zip(photon_energies, coincidence_list)
    }
)
plotthing.fig.gca().get_legend().set_title("Photon Energy")
plotthing

In [ ]:
alpha_bins = sc.linspace(
    "alpha",
    electron_collection.coords["alpha"].nanmin(),
    electron_collection.coords["alpha"].nanmax(),
    100,
)
energy_bins = sc.linspace("energy", 0.0, 10, 100, unit="eV")

In [ ]:
alpha_e_hist_non_masked = electron_collection.drop_masks("close to wiggle").hist(
    energy=energy_bins, alpha=alpha_bins, dim=("p", "pulses", "PE")
)
alpha_e_plot_non_masked = alpha_e_hist_non_masked.plot(norm="log", cmap="PuBuGn")
alpha_e_plot_non_masked

In [ ]:
alpha_e_hist = electron_collection.hist(
    energy=energy_bins, alpha=alpha_bins, dim=("p", "pulses", "PE")
)
alpha_e_plot = alpha_e_hist.plot(norm="log", cmap="PuBuGn")
alpha_e_plot

In [ ]:
plt.figure()
for energy, coin in zip(photon_energies, coincidence_list):
    flat_electron = coin.momenta["e"].flatten(to="hit")
    selected_electrons = flat_electron[~flat_electron.masks["close to wiggle"]]
    electron_energy = selected_electrons.coords["energy"]
    expected_energy = energy - binding_energy
    plt.scatter(
        x=energy.value, y=electron_energy.nanstd(ddof=1).value, label=f"{energy:c} +- {bandwidth:c}"
    )
plt.legend()
plt.ylabel("Electron energy std [eV]")
plt.xlabel("Photon Energy [eV]")
plt.ylim(ymin=0)
plt.show()

In [ ]:
d_p = sc.Dataset(
    data={
        "p_x std": electron_collection.coords["p_x"].sum(dim=["p", "pulses"]),
        "p_y std": electron_collection.coords["p_y"].sum(dim=["p", "pulses"]),
        "p_z std": electron_collection.coords["p_z"].sum(dim=["p", "pulses"]),
    },
    coords={"PE": electron_collection.coords["PE"]},
)
d_energy = sc.Dataset(
    data={"energy std": electron_collection.coords["energy"].sum(dim=["p", "pulses"])},
    coords={"PE": electron_collection.coords["PE"]},
)

In [ ]:
d_energy.plot()

In [ ]:
d_p.plot()

# polarization is 18° to x-axis in x-z-plane